# Acceptance: whole-population force of infection

**This is a sign-off page, not a tutorial.** It checks
`ForceOfInfection(group_by=None)`, which closes the dummy-population half of
composability finding `CX15` (plan `plans/foi-whole-population.plan.md`).

`group_by` is now optional. Left out, the whole population is one group, so the
frequency force of infection is `contact_rate * I_total / N_total` however the
compartments are stratified. That is summer2's default homogeneous mixing.
Before this change it took a dummy one-trait property, stratified onto every
compartment, to say the same thing.

Every section uses an age-stratified SIR model **seeded only in children**. With
that seeding the age groups have different prevalences, so homogeneous mixing
and per-age mixing really do give different epidemics. Nothing here asserts; the
automated checks live in `tests/test_foi_whole_population.py`. Run every cell top
to bottom (a few seconds). Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| A | the compartments carry no extra property |
| B | `group_by=None` gives exactly the epidemic the dummy-property idiom gave |
| C | it is not per-age mixing: adults are infected even though only children were seeded |
| D | the saved force of infection is `contact_rate * I_total / N_total` |

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import FlowModel, Param, Property, PropertyMap, TransitionFlow
from summer4.epi import ForceOfInfection, MixingMatrix

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult"))
params = {"beta": 0.3}
run_kwargs = dict(t0=0.0, t1=120.0, dt=0.25, solver="euler")


def sir(pmap, foi):
    """An SIR model on ``pmap`` whose infection flow uses ``foi``."""
    model = FlowModel(pmap)
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
    return model.compile()


def seeded_in_children(pmap):
    """1,000 children and 1,500 adults; 10 infectious children, no infectious adults."""
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["S"] & age["child"])] = 990.0
    y0[pmap.select(state["I"] & age["child"])] = 10.0
    y0[pmap.select(state["S"] & age["adult"])] = 1500.0
    return y0


def infectious_by_age(result, pmap):
    """Infectious people by age over time, as a date-free DataFrame."""
    comps = np.asarray(result["compartments"].values.data)
    times = np.asarray(result["compartments"].times.values)
    return pd.DataFrame(
        {a: comps[:, pmap.select(state["I"] & age[a])].sum(axis=1) for a in age.traits},
        index=pd.Index(times, name="day"),
    )

## A. The model, with `group_by` left out

The force of infection names only what is infectious and the contact rate. The
compartment map is the plain state × age map: six compartments.

The second model below is the old idiom for comparison: a one-trait `pop`
property stratified onto every compartment, the force of infection grouped by
it, and a `[[1.0]]` mixing matrix. The bar chart compares the compartment
counts, and the table shows both sets of labels.

In [ ]:
pmap = PropertyMap.from_property(state).stratify(age)
whole = sir(pmap, ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("beta")))

pop = Property("pop", ("all",))
dummy_map = pmap.stratify(pop)
dummy = sir(
    dummy_map,
    ForceOfInfection(
        "foi",
        infectious=state["I"],
        group_by=pop,
        mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
        contact_rate=Param("beta"),
    ),
)

pd.Series(
    {"group_by=None": pmap.size, "dummy pop property": dummy_map.size}, name="compartments"
).plot.bar(
    title="A. Compartments in each model",
    labels={"index": "model", "value": "compartments"},
).show()
pd.DataFrame({"group_by=None": pmap.labels(), "dummy pop property": dummy_map.labels()})

> **What to check (A).** Both models have six compartments, and the
> `group_by=None` labels are plain `state=…_age=…`, with no `pop=all` suffix.

## B. Same epidemic as the dummy-property idiom

Run both models from the same start, seeded only in children, and plot the
infectious count by age. The dummy-property lines are dashed and drawn over the
`group_by=None` lines.

In [ ]:
res_whole = whole.run(params, seeded_in_children(pmap), **run_kwargs)
res_dummy = dummy.run(params, seeded_in_children(dummy_map), **run_kwargs)
i_whole = infectious_by_age(res_whole, pmap)
i_dummy = infectious_by_age(res_dummy, dummy_map)

fig = pd.concat(
    [i_whole.add_suffix(": group_by=None"), i_dummy.add_suffix(": dummy pop property")], axis=1
).plot(
    title="B. Infectious people by age: group_by=None and the dummy-property idiom",
    labels={"value": "infectious people", "variable": ""},
)
for trace in fig.data[2:]:
    trace.update(line={"dash": "dash", "width": 3})
fig.show()
pd.Series(
    {"max |group_by=None - dummy| (infectious, any age, any day)": float(
        np.max(np.abs(i_whole.to_numpy() - i_dummy.to_numpy()))
    )},
    name="B",
)

> **What to check (B).** Each dashed line sits exactly on the solid line of the
> same age, and the printed maximum difference is zero or round-off (below
> 1e-3 people).

## C. Not the same as grouping by age without a mixing matrix

Grouping the force of infection by `age` with no mixing matrix is a different
model: each age group catches infection only from its own group
(`I_age / N_age`). It matches homogeneous mixing only while every age group has
the same prevalence. Seeded only in children, the two diverge completely.

In [ ]:
per_age = sir(
    pmap,
    ForceOfInfection("foi", infectious=state["I"], group_by=age, contact_rate=Param("beta")),
)
i_per_age = infectious_by_age(per_age.run(params, seeded_in_children(pmap), **run_kwargs), pmap)

pd.concat(
    [i_whole.add_suffix(": group_by=None"), i_per_age.add_suffix(": group_by=age, no matrix")],
    axis=1,
).plot(
    title="C. Infectious people by age: homogeneous vs per-age mixing",
    labels={"value": "infectious people", "variable": ""},
).show()
pd.DataFrame(
    {
        "group_by=None": i_whole.max(),
        "group_by=age, no matrix": i_per_age.max(),
    }
).rename_axis("age").rename(lambda a: f"peak infectious ({a})")

> **What to check (C).** Under `group_by=None` the adult curve rises (adults
> catch infection from children). Under `group_by=age` with no matrix the adult
> curve stays at zero, and the peak table shows a large adult peak for the first
> and zero for the second.

## D. The saved force of infection

The force of infection is captured under its name, `"foi"`, and saved by
default. With one group it is a plain time series. The plot compares it with
`beta * I_total / N_total` computed by hand from the saved compartments.

In [ ]:
comps = np.asarray(res_whole["compartments"].values.data)
by_hand = params["beta"] * comps[:, pmap.select(state["I"])].sum(axis=1) / comps.sum(axis=1)
saved = np.asarray(res_whole["foi"].values).reshape(-1)
days = np.asarray(res_whole["foi"].times.values)

fig = pd.DataFrame(
    {"saved foi": saved, "beta * I_total / N_total, by hand": by_hand},
    index=pd.Index(days, name="day"),
).plot(
    title="D. The saved force of infection and the hand computation",
    labels={"value": "force of infection (per day)", "variable": ""},
)
fig.data[1].update(line={"dash": "dash", "width": 3, "color": "black"})
fig.show()
pd.Series(
    {
        "saved output dims": res_whole["foi"].dims,
        "max |saved - by hand|": float(np.max(np.abs(saved - by_hand))),
    },
    name="D",
    dtype=object,
)

> **What to check (D).** The dashed hand computation lies on the saved
> series all the way through, the output's dims are `('time',)`, and the
> maximum difference is round-off (below 1e-6).